<Так как после многократных попыток так и не удалось установить Airflow и докер, задание было выполнено в Jupiter Notebook>

In [4]:
# Проверяем Airflow
import airflow
print(f"Airflow версия: {airflow.__version__}")

# Проверяем доступные модули
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.empty import EmptyOperator
from airflow.utils.dates import days_ago

print("Импорты работают")

Airflow версия: 2.7.1
Импорты работают


<Jupiter Notebook видит, что Airflow установлен, но полноценный функционал Airflow воспроизвести не удается. Код в ячейке ниже представлен для Airflow>

In [24]:
from datetime import datetime
from airflow import DAG
from airflow.operators.python import PythonOperator
from airflow.operators.empty import EmptyOperator
import pandas as pd
import sqlite3


def load_customers():
    df = pd.read_csv(r"C:\Users\anastasia.nagoliuk\Desktop\customer.csv")
    df['DOB'] = pd.to_datetime(df['DOB'], errors='coerce')
    conn = sqlite3.connect(r"C:\Users\anastasia.nagoliuk\Desktop\retail.db")
    df.to_sql('customer', conn, if_exists='replace', index=False)
    conn.close()


def load_products():
    df = pd.read_csv(r"C:\Users\anastasia.nagoliuk\Desktop\product.csv")
    df['list_price'] = pd.to_numeric(df['list_price'], errors='coerce')
    df['standard_cost'] = pd.to_numeric(df['standard_cost'], errors='coerce')
    conn = sqlite3.connect(r"C:\Users\anastasia.nagoliuk\Desktop\retail.db")
    df.to_sql('product', conn, if_exists='replace', index=False)
    conn.close()


def load_orders():
    df = pd.read_csv(r"C:\Users\anastasia.nagoliuk\Desktop\orders.csv")
    df['order_date'] = pd.to_datetime(df['order_date'], errors='coerce')
    conn = sqlite3.connect(r"C:\Users\anastasia.nagoliuk\Desktop\retail.db")
    df.to_sql('orders', conn, if_exists='replace', index=False)
    conn.close()


def load_order_items():
    df = pd.read_csv(r"C:\Users\anastasia.nagoliuk\Desktop\order_items.csv")
    df['quantity'] = pd.to_numeric(df['quantity'], errors='coerce')
    df['item_list_price_at_sale'] = pd.to_numeric(df['item_list_price_at_sale'], errors='coerce')
    df['item_standard_cost_at_sale'] = pd.to_numeric(df['item_standard_cost_at_sale'], errors='coerce')
    conn = sqlite3.connect(r"C:\Users\anastasia.nagoliuk\Desktop\retail.db")
    df.to_sql('order_items', conn, if_exists='replace', index=False)
    conn.close()


with DAG(
    dag_id='retail_etl_dag',
    description='Ежедневная загрузка данных о клиентах продуктах и заказах',
    schedule='@daily',
    start_date=datetime(2024, 1, 1),
    catchup=False,
) as dag:
    
    start_task = EmptyOperator(task_id='start')
    
    load_customers_task = PythonOperator(
        task_id='load_customers',
        python_callable=load_customers,
    )
    
    load_products_task = PythonOperator(
        task_id='load_products',
        python_callable=load_products,
    )
    
    load_orders_task = PythonOperator(
        task_id='load_orders',
        python_callable=load_orders,
    )
    
    load_order_items_task = PythonOperator(
        task_id='load_order_items',
        python_callable=load_order_items,
    )
    
    end_task = EmptyOperator(task_id='end')
    
    
    start_task >> [load_customers_task, load_products_task, load_orders_task, load_order_items_task] >> end_task